# E. 독립변수 16개(t-test + logistic regression) QSVM(양자 커널 SVM) 버전

## 1. 라이브러리 설치

In [ ]:
# Colab에서 실행할 때만 아래 주석을 풀고, 프로젝트 폴더 경로를 맞춰 주세요.
# (로컬 Jupyter / VS Code에서는 이 노트북이 있는 폴더에서 그대로 실행)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<프로젝트 폴더>

## 2. 데이터 불러오기 및 탐색 함수

In [ ]:
import numpy as np
import pandas as pd

from config import (
    SEEDS, QUBIT_RANGE, SELECTION_METHODS, ENCODINGS, REDUCTIONS,
)
from src.common import (
    load_raw, run_eda, preprocess, build_df16, split16, iterate_splits,
)
from config import QSVM_GAMMAS
from src.qsvm import encode, qsvm_sweep, classical_experiment
from src.metrics import save_result, save_sweep, summarize_results

NOTEBOOK = "ind16_QSVM"



# 데이터: data/FILE2_중공업.xls (config.DATA_PATH)
df = load_raw()
df.tail(5)

## 3. 전체 탐색 실행

In [ ]:
# 탐색 함수와 출력 내용은 src/common.py 의 run_eda
x, dates, tables, summary = run_eda(df)

## 4. 전처리

### 기존 전처리

In [ ]:
# 전처리 클래스와 파이프라인은 src/common.py 의 preprocess
prep = preprocess(df)

### 독립변수 16개 추출

In [ ]:
# 148개 전처리 결과에서 16개 변수만 뽑아 df16 만들기 (src/common.py 의 build_df16)
df16 = build_df16(prep).df16
df16.head()

#### 데이터 분할

In [ ]:
# 기준 seed(config.SEED) 분할 확인: 148개 버전과 같은 분할인지 내부에서 검증
# (실험 셀에서는 seed마다 prepare_split 으로 다시 나누고 변수도 다시 고른다)
sp = split16(df16, prep)

## 6. QSVM (양자 커널 SVM)

### 구성

VQC와 달리 **양자 회로의 파라미터를 학습하지 않는다.** 회로로 샘플 간 유사도(커널)만 계산하고, 분류는 고전 SVM이 한다.

**양자 인코딩** (`config.ENCODINGS`, 양자 모델에만 적용)

| 인코딩 | 양자 상태 | 입력 수 d (n큐비트) |
|---|---|---|
| basic | 기존 방식: QSVM은 IQP 특징 맵, VQC는 RY 각도 | n |
| angle | 큐비트마다 RY(θ), 얽힘 없음 | n |
| iqp | H → Z·ZZ 위상 반복 (인접 큐비트 얽힘), QSVM·VQC 공통 | n |
| dense_angle | 큐비트 하나에 변수 2개: RY(θ₁) 후 RZ(θ₂) | 2n (n ≤ 8) |
| amplitude | 입력 벡터를 정규화해 상태 진폭으로 사용 | 2ⁿ (n ≤ 4) |

**입력 차원 맞추기** (`config.REDUCTIONS`): 선택된 변수 16개를 d개로 줄인다.

| 방법 | 처리 |
|---|---|
| pca | 표준화 → PCA d개 성분 → 성분 표준화 (기존 방식) |
| select | 변수 선택 순위 상위 d개 → 표준화 |

- d = 16이면 줄이지 않고 표준화만 한다. 각도형 인코딩은 θ = π·tanh(z/2)로 바꾼다.
- 쓸 수 없는 큐비트 수(예: amplitude 5큐비트 이상)는 자동으로 건너뛴다.
- basic + pca에서 n = 8은 기존 8큐비트(PCA 8개), n = 16은 기존 16큐비트 입력과 같다.
- 고전 모델은 PCA 없이 선택된 변수 16개를 그대로 쓴다.

| 공통 단계 | 구성 |
|---|---|
| 양자 커널 | K(a, b) = abs(⟨ψ(a)\|ψ(b)⟩)², 상태를 NumPy로 정확히 계산 (PennyLane 불필요) |
| 분류기 | scikit-learn SVC (precomputed kernel) |
| 튜닝 | 각도 스케일 γ(`config.QSVM_GAMMAS`)와 C를 **검증셋 ROC-AUC**로 선택, 테스트셋은 최종 평가에만 사용 |
| 고전 기준선 | 선택된 변수 16개(PCA 없음)에 RBF-SVM, 로지스틱 회귀 |

- iqp 특징 맵은 큐비트가 많을수록 커널 값이 0 근처로 몰리는 현상(커널 집중)이 심해서 γ를 작게 잡아야 한다. 선택된 γ나 C가 탐색 범위의 끝이면 경고가 출력된다.
- iqp 16큐비트는 상태벡터가 65,536차원이라 γ 하나당 수십 초가 걸리고 메모리를 약 1~2GB 쓴다. angle / dense_angle은 곱 상태라 훨씬 빠르다.


### 실험 설계: 변수 선택 × 인코딩 × 큐비트 수 × seed

`config.py`의 목록을 모두 조합해 반복한다.

| 설정 | 내용 | 기본값 |
|---|---|---|
| `SELECTION_METHODS` | 변수 선택 방법 (seed마다 **훈련셋에서만** 선택) | `["fixed16"]` |
| `ENCODINGS` | 양자 인코딩 (basic / angle / iqp / dense_angle / amplitude) | `["basic"]` |
| `REDUCTIONS` | 큐비트보다 입력이 많을 때 줄이는 방법 (pca / select) | `["pca"]` |
| `QUBIT_RANGE` | 큐비트 수 | 1 ~ 16 |
| `SEEDS` | 반복 seed (분할 · 전처리 · 선택 · 모델 난수가 모두 바뀜) | 5개 |

seed마다 다시 하는 것: train/valid/test 분할(60/20/20, 층화) → 전처리 fit → 변수 후보(148개 + 업력) → 변수 선택 → 인코딩 fit → 모델 학습.

- 결과는 조합 하나가 끝날 때마다 `results/results.csv`에 저장되므로, 중간에 멈춰도 그때까지의 결과는 남는다.
- 마지막 셀에서 **평균 ± 표준오차**(95% 신뢰구간 포함)로 요약한다.
- 실행 시간은 조합 수에 비례한다. 먼저 목록을 줄여서 돌려 본다 (예: `SEEDS = [42]`, `QUBIT_RANGE = [4, 8, 16]`).
- `fixed16`은 전체 데이터로 고른 기존 목록이라 테스트 정보가 섞여 있다. 다른 방법(훈련셋에서만 선택)과의 차이가 선택 편향의 크기다.


### QSVM (변수 선택 × 인코딩 × 1~16큐비트 × seed)


In [ ]:
# 인코딩 · 양자 커널 · run_qsvm 은 src/qsvm.py
for data in iterate_splits(df, SELECTION_METHODS, SEEDS):
    for encoding in ENCODINGS:
        for reduction in REDUCTIONS:
            summary, runs = qsvm_sweep(
                QUBIT_RANGE, data.X, data.Y, gammas=QSVM_GAMMAS,
                encoding=encoding, reduction=reduction, seed=data.seed,
            )

            save_sweep(
                NOTEBOOK, "QSVM", runs, seed=data.seed,
                selection=data.selection, encoding=f"{encoding}/{reduction}",
                params_of=lambda run: {
                    "gamma": run["info"]["gamma"], "C": run["info"]["C"],
                    "features": data.selected,
                },
            )

### 고전 기준선 (선택된 변수 16개, PCA 없음, × seed)


In [ ]:
for data in iterate_splits(df, SELECTION_METHODS, SEEDS):
    # 고전 모델은 PCA 없이 선택된 변수 16개 (16큐비트 QSVM과 같은 입력각)
    enc16 = encode(16, *data.X, *data.Y, method="basic", verbose=False, seed=data.seed)

    classical16_results = classical_experiment(
        f"변수 16개, {data.selection}, seed {data.seed}", enc16.angles,
        tuple(np.asarray(part) for part in data.Y), seed=data.seed,
    )

    for name, row in classical16_results.iterrows():
        save_result(
            NOTEBOOK, f"{name.split(' (')[0]} [변수 16개]", row, seed=data.seed,
            selection=data.selection, params={"model": name, "features": data.selected},
        )

## 모델 성능 비교

In [ ]:
# ============================================================
# 요약: 평균 ± 표준오차 (테스트셋)
# ============================================================

summary_table, summary_full = summarize_results(notebook=NOTEBOOK)

print(f"[{NOTEBOOK}: 테스트 성능, 평균 ± 표준오차]")
print(summary_table.to_string())

# 지표별 n, mean, std, se, 95% 신뢰구간 전체 표
summary_full.round(4)

In [ ]:
# 참고: 다른 노트북 결과 (results/results.csv, 테스트셋, 평균 ± 표준오차)
reference_table, _ = summarize_results(exclude_notebook=NOTEBOOK)

print("[참고: 다른 노트북 결과]")
print(reference_table.to_string())

## 7. 연구 보고서

> **참고:** 아래 보고서는 8큐비트와 16큐비트만, seed 하나로 실험하던 **이전 실험(리팩터링 전)** 기록이다.
> 현재 코드는 변수 선택 방법 × 인코딩 × 1~16큐비트 × 여러 seed로 실험한다.


2026년 11월 3일

#### 1. 실험 개요

| 항목 | 설정 |
|---|---|
| 데이터 | FILE2_중공업 (3,140개 기업, 산업 기준 1:1 매칭) |
| Train / Validation / Test | 1,884 / 628 / 628 (60% / 20% / 20%, 층화추출, seed 42) |
| 독립변수 | 16개 (t-test + logistic regression으로 도출) |
| 타깃 변수 | 정상/폐업 (이진분류) |
| 클래스 비율 | 50:50 |
| 양자회로 | 8큐비트 (PCA 8개 입력), 16큐비트 (변수 16개 직접 입력) |
| PCA 8개 누적 설명분산 비율 | 74.62% |
| 목적 | 16개 변수에서 QSVM의 성능을 VQC, 고전 모델, XGBoost와 비교하고 큐비트 수에 따른 변화 확인 |

※ 데이터 분할과 전처리는 기존 16개 변수 실험(ind16)과 동일함. 업력(K1)이 포함되어 있다.

**독립변수 16개 구성**

1. 영업이익률
2. 재료비/매출액
3. 감가상각률
4. 금융비용대총부채비율
5. 금융비용대총비용비율
6. EBITDA/총차입금
7. 차입금의존도
8. 장단기대여금/총자산
9. 법인세부담률
10. 순운전자본대총자본비율
11. 유동자산구성비율
12. 유보액대총자산비율
13. 재고자산대유동자산비율
14. 매입채무회전기간
15. 자본금회전율(회)
16. 업력

#### 2. 모델 구성

**QSVM 입력 처리 (훈련셋에서만 fit, 검증·테스트는 transform만 적용)**

| 구분 | 8큐비트 | 16큐비트 |
|---|---|---|
| 입력 변수 | 16개 | 16개 |
| 차원 축소 | PCA 8개 성분 | 없음 |
| 큐비트 대응 | 성분 1개 = 큐비트 1개 | 변수 1개 = 큐비트 1개 |
| 입력 처리 | VarianceThreshold → StandardScaler → PCA → 성분 StandardScaler → θ = π·tanh(z/2) | VarianceThreshold → StandardScaler → θ = π·tanh(z/2) |

**QSVM 공통 설정**

| 항목 | 설정 |
|---|---|
| 특징 맵 | IQP/ZZ 계열, \|ψ(x)⟩ = D(x) H⊗ⁿ D(x) H⊗ⁿ \|0⟩ (반복 2회) |
| D(x) | exp(i[Σ γxᵢzᵢ + Σ γxᵢxᵢ₊₁zᵢzᵢ₊₁]) (인접 큐비트 ZZ 얽힘) |
| 커널 | 충실도 커널 \|⟨ψ(a)\|ψ(b)⟩\|² |
| 분류기 | SVC (kernel="precomputed") |
| 하이퍼파라미터 | gamma(각도 배율), C를 검증셋 ROC-AUC로 선택 |
| gamma 탐색 범위 | 8큐비트 0.03, 0.05, 0.1, 0.25 / 16큐비트 0.05, 0.1, 0.2, 0.3 |
| 선택된 설정 | 8큐비트 기입 / 16큐비트 기입 |
| 시뮬레이터 | NumPy 상태벡터 (노이즈 없음, shots 없음) |

**고전 기준선 (QSVM과 같은 입력)**

| 입력 | RBF-SVM (검증 AUC로 선택) | 로지스틱 회귀 |
|---|---|---|
| PCA 8개 | gamma = 0.1, C = 1 | 사용 |
| 변수 16개 | gamma = 0.01, C = 10 | 사용 |

#### 3. 모델별 성능 비교 (테스트셋)

**A. 8큐비트 (PCA 8개 입력)**

| 지표 | QSVM (8q) | RBF-SVM | 로지스틱 회귀 |
|---|---:|---:|---:|
| Accuracy | **0.8376** | 0.8344 | 0.8312 |
| Balanced accuracy | **0.8376** | 0.8344 | 0.8312 |
| Precision | 0.8155 | 0.8281 | 0.8230 |
| Recall | **0.8726** | 0.8439 | 0.8439 |
| F1-score | **0.8431** | 0.8360 | 0.8333 |
| ROC-AUC | **0.8973** | 0.8945 | 0.8843 |
| AP | 0.8680 | **0.8692** | 0.8532 |
| Log Loss | **0.3983** | 0.4027 | 0.4275 |

**B. 16큐비트 (변수 16개 직접 입력)**

| 지표 | QSVM (16q) | RBF-SVM | 로지스틱 회귀 |
|---|---:|---:|---:|
| Accuracy | **0.8583** | **0.8583** | 0.8328 |
| Balanced accuracy | **0.8583** | **0.8583** | 0.8328 |
| Precision | **0.8399** | 0.8378 | 0.8276 |
| Recall | 0.8854 | **0.8885** | 0.8408 |
| F1-score | 0.8620 | **0.8624** | 0.8341 |
| ROC-AUC | **0.9268** | 0.9248 | 0.8986 |
| AP | **0.9217** | 0.9208 | 0.8759 |
| Log Loss | **0.3419** | 0.3480 | 0.4077 |

※ Precision, Recall, F1-score는 원본 타깃 코드 2를 양성 클래스로 계산.

**C. 기존 16개 변수 실험(ind16)과 비교**

| 모델 | 입력 | Accuracy | ROC-AUC | Log Loss |
|---|---|---:|---:|---:|
| XGBoost | 변수 16개 | **0.8615** | **0.9453** | **0.3044** |
| QSVM (16큐비트) | 변수 16개 | 0.8583 | 0.9268 | 0.3419 |
| QSVM (8큐비트) | PCA 8개 | 0.8376 | 0.8973 | 0.3983 |
| PCA-XGBoost | PCA 8개 | 0.8232 | 0.8946 | 0.4052 |
| DR-VQC (8큐비트) | PCA 8개 | 0.7532 | 0.8313 | 0.5128 |
| VQC (8큐비트) | PCA 8개 | 0.6990 | 0.7791 | 0.5619 |

#### 4. 핵심 결과

- 8큐비트 QSVM의 ROC-AUC는 0.8973으로, 같은 입력의 VQC(0.7791)보다 0.118, DR-VQC(0.8313)보다 0.066 높다.
- 8큐비트 QSVM은 PCA-XGBoost(0.8946)와 같은 수준이다.
- 16큐비트 QSVM의 ROC-AUC는 0.9268로, 8큐비트(0.8973)보다 0.030 높고 XGBoost(0.9453)에는 0.019 못 미친다. Accuracy는 XGBoost와 0.0032 차이다.
- 두 경우 모두 같은 입력의 RBF-SVM과 거의 같다. 8큐비트는 0.0028, 16큐비트는 0.0020 차이다.
- 로지스틱 회귀보다는 8큐비트에서 0.013, 16큐비트에서 0.028 높다.

#### 5. 해석

**VQC와 비교**

- 같은 PCA 8개 입력에서 QSVM이 VQC와 DR-VQC보다 높다. 회로를 학습하지 않는 QSVM이 더 잘 나왔으므로, VQC의 낮은 성능은 양자 특징 공간의 한계보다 회로 학습 쪽 문제일 가능성이 있다. 이번 실험만으로 확정할 수는 없다.

**고전 모델과 비교**

- QSVM과 RBF-SVM의 차이는 테스트셋 628개 기준 AUC 표준오차(약 0.015)보다 훨씬 작다. 양자 커널의 우위는 확인되지 않는다.

**8큐비트와 16큐비트 비교**

- 16큐비트의 향상(+0.030)은 큐비트 수가 늘어서인지, PCA를 쓰지 않아 정보 손실이 없어서인지 구분되지 않는다. 8큐비트 입력은 PCA로 설명분산의 74.62%만 유지한다.
- 같은 입력 변화에서 RBF-SVM도 0.8945에서 0.9248로 비슷하게(+0.030) 올랐다. 향상의 원인은 양자 회로보다 입력 정보량일 가능성이 크다.
- 16큐비트의 RBF-SVM gamma(0.01)가 탐색 범위의 하단이므로, 더 작은 값에서 고전 모델이 더 나아질 여지가 있다.

**XGBoost와 비교**

- 16큐비트 QSVM은 XGBoost에 거의 근접했지만 AUC 0.019, Log Loss 0.037 낮다. 이 차이는 표준오차 정도라 우열을 단정하기 어렵다.

#### 6. 결론

독립변수 16개를 사용한 QSVM은 8큐비트(PCA 8개)에서 ROC-AUC 0.8973, 16큐비트(변수 직접 입력)에서 0.9268을 기록해 같은 입력의 VQC와 DR-VQC를 크게 앞섰고 XGBoost에 근접했다. 다만 같은 입력의 고전 RBF-SVM과 차이가 없고, 16큐비트의 향상은 PCA 제거에 따른 정보 보존으로도 설명되므로 양자 모델의 우위를 확인했다고 보기는 어렵다.

#### 7. 한계 및 유의사항

- 시뮬레이터 결과이며, 실제 양자 하드웨어의 노이즈와 shots 오차는 반영되지 않았다.
- 한 번의 분할(seed 42)과 한 번의 실행 결과이므로 시드 변동은 확인하지 못했다.
- 16개 변수에는 업력(K1)이 포함되어 있어, 업력의 산출 시점에 따른 데이터 누수 가능성이 남아 있다.
- 8큐비트와 16큐비트는 큐비트 수와 입력 방식(PCA 유무)이 함께 달라서 큐비트 수만의 효과는 분리되지 않는다.
- 비교표의 XGBoost, VQC, DR-VQC 값은 ind16 보고서의 값이다.

#### 8. 추가 확인 사항

- [ ] 8큐비트 입력을 PCA가 아닌 다른 방식(예: 중요 변수 8개 선택)으로 만들어 입력 방식 효과 분리
- [ ] PCA 16개 입력의 16큐비트 QSVM으로 PCA 유무와 큐비트 수 효과 분리
- [ ] 업력 제외 후 QSVM 성능 비교
- [ ] 고전 RBF-SVM의 gamma 탐색 범위를 더 낮은 값까지 확장
- [ ] 여러 랜덤 시드와 부트스트랩 신뢰구간으로 모델 간 AUC 차이 검증
- [ ] 16큐비트 VQC, DR-VQC는 학습 시간 문제로 이번에 제외했으며, 필요하면 별도 환경에서 재시도